# 🏟️ Member 4: Single-Stage Cluster Sampling & Unified Design Comparison
**Project:** Design, Estimation and Efficiency Comparison of Sampling Techniques on IPL Batting Data (2008–2024)

### Objectives:
1. Implement single-stage cluster sampling with clusters = matches ($M = 1,095$ clusters).
2. Compute the Ratio Estimator for mean $\bar{y}_c$ and unbiased estimator for total.
3. Calculate Intraclass Correlation Coefficient (ICC, $\rho$) from ANOVA mean squares.
4. Construct the unified comparison table across all 6 sampling designs.
5. Review Monte Carlo simulation (500 reps) results: Bias, SE, RMSE, and 95% CI coverage.
6. Conduct F-tests on design variance ratios.


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import config
from sampling.frame import load_frame
from sampling import cluster, estimators as est
from sampling.comparison import build_comparison_table
from sampling.simulation import summarise_simulation
from sampling.tests import anova_clusters, f_test_variance_ratio

frame = load_frame()
print(f"Loaded frame with {len(frame):,} rows and {frame['match_id'].nunique():,} match clusters.")


## 1. Cluster Sampling & Ratio Estimation
Because clusters have unequal sizes $m_i$, we use the ratio estimator:
$$\bar{y}_c = \frac{\sum_{i=1}^m y_i}{\sum_{i=1}^m m_i}$$


In [ ]:
s_cl, m_actual = cluster.draw_cluster_sample(frame, rng=np.random.default_rng(config.SEED + 5))
res_cl = cluster.analyse_cluster(s_cl, frame, m_actual, "match_id", config.N,
                                 float(frame["runs"].mean()), float(frame["runs"].sum()), float((frame["runs"] >= 30).mean()))

print(f"Clusters Drawn: {m_actual} matches, Total units n_actual: {res_cl['n_actual']:,}")
print(f"Cluster Ratio Mean: {res_cl['mean']:.4f}, SE: {res_cl['se_mean']:.4f}")
print(f"95% CI: ({res_cl['ci_mean'][0]:.2f}, {res_cl['ci_mean'][1]:.2f})")
print(f"Unbiased Total: {res_cl['total']:,.0f}, SE: {res_cl['se_total']:,.0f}")


## 2. Intraclass Correlation (ICC) and Design Effect
From ANOVA mean squares:
$$\hat{\rho} = \frac{MSB - MSW}{MSB + (\bar{M}_0 - 1) MSW}, \quad Deff = 1 + (\bar{M}_0 - 1) \hat{\rho}$$


In [ ]:
icc_match = cluster.compute_icc(frame, "match_id")
print("=== Match-Level Clustering ===")
print(f"ICC (ρ): {icc_match['ICC_rho']:.4f}")
print(f"Design Effect (Deff): {icc_match['design_effect']:.2f}")
print(f"MSB: {icc_match['MSB']:.4f}, MSW: {icc_match['MSW']:.4f}")
print(icc_match["interpretation"])

icc_venue = cluster.compute_venue_icc(frame)
print("\n=== Venue-Level Clustering (Contrast) ===")
print(f"ICC (venue): {icc_venue['ICC_rho']:.4f}")


## 3. Master Cross-Design Comparison Table


In [ ]:
comp_table = build_comparison_table(frame, save=False)
display_cols = ["design", "n", "mean", "se_mean", "ci_lower", "ci_upper", "proportion", "RE_vs_SRS", "abs_error_mean"]
comp_table[display_cols]


## 4. Monte Carlo Simulation Summary (500 Replications)


In [ ]:
sim_df = pd.read_parquet("../results/simulation.parquet")
sim_summary = summarise_simulation(sim_df, float(frame["runs"].mean()), float((frame["runs"] >= 30).mean()))
sim_summary[["design", "emp_mean_of_means", "bias_mean", "se_mean", "rmse_mean", "coverage_mean", "RE_mean_vs_SRS"]]


## 5. Design Variance F-Tests


In [ ]:
f_test = f_test_variance_ratio(
    comp_table[comp_table["design"]=="SRSWOR"]["var_mean"].iloc[0],
    comp_table[comp_table["design"]=="Stratified-Prop"]["var_mean"].iloc[0],
    config.n - 1, config.n - 1,
    "SRSWOR", "Stratified-Prop"
)
print(f"F-test: {f_test['test']}")
print(f"F-value: {f_test['F_value']:.4f}, p-value: {f_test['p_value']:.4e}")
print(f"Decision: {f_test['decision']}")
